# CI/CD Pipline for DDOS detection

This notebook implements a complete end-to-end Pipeline with the following steps:
1. Feature Engineering  
   Raw input data stored in S3 is cleaned, normalized, encoded and selected top 10 features to train the model. The output is written back to S3 in CSV format.

2. Feature Store Ingestion
   A new Feature Group is created, and the cleaned data is ingested into the Feature Store to be used in downstream tasks such as splitting into training, validation, and test sets.

3. .....

In [1]:
!pip install sagemaker

In [2]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.parameters import ParameterInteger
from sagemaker.workflow.parameters import ParameterFloat
from sagemaker.workflow.parameters import ParameterString
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor, ScriptProcessor
from sagemaker.workflow.pipeline_context import PipelineSession
import sys

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## 1. Setup

In [3]:
!mkdir -p code

In [4]:
framework_version = "1.2-1"

In [5]:
# SageMaker session and role
sagemaker_session = sagemaker.session.Session()
region = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()

In [6]:
# define paths
s3_raw_data_prefix = 'final_project/data'
data_filename= 'DDoS-Friday-no-metadata.parquet'
s3_engineered_prefix = 'final_project/feature_engineer'
engineered_filename = 'engineered_features.csv'
input_uri = f"s3://{bucket}/{s3_raw_data_prefix}"

## 2. Create Steps

### 2.1 Feature engineering 

#### 2.1.1 Pipeline Parameters

In [7]:
input_data_uri = f"s3://{bucket}/{s3_raw_data_prefix}/{data_filename}"
output_data_uri = f"s3://{bucket}/{s3_engineered_prefix}"

In [8]:
# Pipeline Parameters
input_data = ParameterString(name="InputData", default_value=input_data_uri)
output_data = ParameterString(name="OutputData", default_value=output_data_uri)

#### 2.1.2. Processor setup

In [9]:
sklearn_processor = SKLearnProcessor(
    framework_version=framework_version,
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    base_job_name="ddos-traffic-pipeline-processor",
    sagemaker_session=sagemaker_session
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


#### 2.1.3. Step script

In [10]:
%%writefile code/feature_engineering.py
print("code/feature_engineering.py")
import pandas as pd
import numpy as np
import boto3
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import sys
import os
import io
import uuid
import time
from time import gmtime, strftime, sleep

if __name__ == "__main__":
    base_dir = "/opt/ml/processing"
    df = pd.read_parquet(f'{base_dir}/input/DDoS-Friday-no-metadata.parquet')
    print(df.shape)
    
    # shuffel the data
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(df['label'].value_counts())
    
    # Drop columns with all NaNs
    df_cleaned = df.dropna(axis=1, thresh=1)
    
    # Binary label encoding
    df_cleaned['label'] = df_cleaned['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)
    print(df_cleaned.shape)
    
    # Normalize numeric features
    features = df.drop('label', axis=1)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)
    X_scaled_df = pd.DataFrame(X_scaled, columns=features.columns)
    X_scaled_df['label'] = df['label']
    
    # Feature selection using Random Forest
    X = X_scaled_df.drop('label', axis=1)
    y = X_scaled_df['label']
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X, y)
    
    # Select top 10 features
    importances = clf.feature_importances_
    top_features = X.columns[np.argsort(importances)[::-1][:10]]
    
    # df_final dataframe with final selected features will be stored. df_final can be used by the model to train/validate and test.
    df_final = df_cleaned[top_features.tolist() + ['label']]
    
    print(top_features)
    
    current_time_sec = int(round(time.time()))
    
    # Move target column to front
    target_col = "label"
    df_final = df_final[[target_col] + [col for col in df_final.columns if col != target_col]]
    
    # Add required metadata columns
    record_identifier_feature_name = "record_id"
    df_final[record_identifier_feature_name] = [str(uuid.uuid4()) for _ in range(len(df_final))]
    event_time_feature_name = 'event_time'
    df_final[event_time_feature_name] = pd.Series([current_time_sec] * len(df_final), dtype="float64")
    pd.DataFrame(df_final).to_csv(f"{base_dir}/feature_engineered/engineered_features.csv", header=True, index=False)   

Overwriting code/feature_engineering.py


In [11]:
input_data_uri

's3://sagemaker-us-east-1-249645693565/final_project/data/DDoS-Friday-no-metadata.parquet'

#### 2.1.4. Processing Step 

In [12]:
step_feature_engineering_process = ProcessingStep(
    name="FeatureEngineering",
    processor=sklearn_processor,
    inputs=[ProcessingInput(source=input_data_uri, destination="/opt/ml/processing/input")],
    outputs=[ProcessingOutput(output_name="feature_engineered", source="/opt/ml/processing/feature_engineered",
                              destination = output_data_uri)],
    code= "code/feature_engineering.py",
)

### 2.2 Feature store ingestion

#### 2.2.1. Pipeline Parameters

In [13]:
#input_feature_uri = f"s3://{bucket}/{s3_engineered_prefix}/{engineered_filename}"
input_feature_uri = step_feature_engineering_process.properties.ProcessingOutputConfig.Outputs["feature_engineered"].S3Output.S3Uri

In [14]:
input_feature_uri

{'_step': <sagemaker.workflow.steps.ProcessingStep object at 0x7f0ec946f260>, 'step_name': 'FeatureEngineering', 'path': "ProcessingOutputConfig.Outputs['feature_engineered'].S3Output.S3Uri", '_shape_names': ['S3Uri'], '__str__': 'S3Uri'}

In [15]:
from time import strftime, gmtime
group_name = f"feature-group-ddos-detection-{strftime('%d-%H-%M-%S', gmtime())}"

feature_group_name = ParameterString(name="FeatureGroupName", default_value=group_name)
sagemaker_role = role

#### 2.2.2. Step Script

In [16]:
%%writefile 'code/feature_store_ingest.py'
import argparse
import boto3
import sagemaker
from sagemaker import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import time
from time import gmtime, strftime, sleep
import pandas as pd

def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--feature-group-name", type=str, required=True)
    parser.add_argument("--sagemaker-role", type=str, required=True)
    args = parser.parse_args()
    intrusion_feature_group_name = args.feature_group_name
    role = args.sagemaker_role
    print(intrusion_feature_group_name, role)
    base_dir = "/opt/ml/processing"
    # intrusion_feature_group_name = "feature-group-DDOS-detection" + strftime("%d-%H-%M-%S", gmtime())
    # Create boto3 session
    boto_session = boto3.Session(region_name="us-east-1") 
    
    # Create the SageMaker session
    sagemaker_session = Session(boto_session=boto_session)
    region = sagemaker_session.boto_region_name
    bucket = sagemaker_session.default_bucket()
    
    sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
    featurestore_runtime = boto_session.client(
        service_name="sagemaker-featurestore-runtime", region_name=region
    )
    
    feature_store_session = Session(
        boto_session=boto_session,
        sagemaker_client=sagemaker_client,
        sagemaker_featurestore_runtime_client=featurestore_runtime,
    )
    
    intrusion_feature_group = FeatureGroup(
        name=intrusion_feature_group_name, sagemaker_session=feature_store_session
    )
    df_final = pd.read_csv(f'{base_dir}/input/engineered_features.csv')
    intrusion_feature_group.load_feature_definitions(data_frame=df_final)
   
    intrusion_feature_group.create(
        s3_uri= f's3://{bucket}/final_project/feature-store/{intrusion_feature_group_name}/',
        record_identifier_name= "record_id",
        event_time_feature_name= 'event_time',
        role_arn=role,
        enable_online_store=True,    
    )
    
    wait_for_feature_group_creation_complete(feature_group=intrusion_feature_group)
    intrusion_feature_group.describe()
    print("Ingesting data into Feature Store.")
    intrusion_feature_group.ingest(data_frame=df_final, max_workers=3, wait=True)
    print("Ingestion complete.")

Overwriting code/feature_store_ingest.py


#### 2.2.3. Processing step

In [17]:
script_processor = ScriptProcessor(
    image_uri=f"885854791233.dkr.ecr.{region}.amazonaws.com/sagemaker-distribution-prod:1-cpu",
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    command=["python3"],
    sagemaker_session=sagemaker_session
)

In [18]:
step_feature_store_process = ProcessingStep(
    name="FeatureIngest",
    processor=script_processor,
    inputs=[ProcessingInput(source=input_feature_uri, destination="/opt/ml/processing/input")], 
    outputs=[ProcessingOutput(source="/opt/ml/processing/output", destination=output_data_uri)], 
    code= "code/feature_store_ingest.py",
    job_arguments=[
        "--feature-group-name", feature_group_name,
        "--sagemaker-role", sagemaker_role
    ]
)

In [19]:
input_feature_uri

{'_step': <sagemaker.workflow.steps.ProcessingStep object at 0x7f0ec946f260>, 'step_name': 'FeatureEngineering', 'path': "ProcessingOutputConfig.Outputs['feature_engineered'].S3Output.S3Uri", '_shape_names': ['S3Uri'], '__str__': 'S3Uri'}

In [20]:
feature_group_name

ParameterString(name='FeatureGroupName', parameter_type=<ParameterTypeEnum.STRING: 'String'>, default_value='feature-group-ddos-detection-13-22-14-45')

### 2.3 Train/Test/Val Split Step

In [21]:
%%writefile 'code/data_split.py'
import os
import pandas as pd
from sklearn.model_selection import train_test_split

if __name__ == "__main__":
    BASE_DIR = "/opt/ml/processing"
    df = pd.read_csv(os.path.join(BASE_DIR, "input", "engineered_features.csv"))

    for col in ["record_id", "event_time"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    X = df.drop(columns=["label"])
    y = df["label"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, random_state=1, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=1, stratify=y_temp
    )

    def save_split(features, labels, folder, filename):
        path = os.path.join(BASE_DIR, folder)
        os.makedirs(path, exist_ok=True)
        pd.concat([labels, features], axis=1).to_csv(
            os.path.join(path, filename), index=False, header=False
        )

    save_split(X_train, y_train, "train", "train.csv")
    save_split(X_val,   y_val,   "validation", "validation.csv")
    save_split(X_test,  y_test,  "test", "test.csv")

Overwriting code/data_split.py


In [22]:
split_processor = SKLearnProcessor(
    framework_version=framework_version,
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    base_job_name="ddos-traffic-pipeline-splitter",
    sagemaker_session=sagemaker_session
)

split_output_prefix = "final_project/splits"
step_split_process = ProcessingStep(
    name="TrainValTestSplit",
    processor=split_processor,
    inputs=[
        ProcessingInput(
            source=input_feature_uri,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{bucket}/{split_output_prefix}/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation",
            destination=f"s3://{bucket}/{split_output_prefix}/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{bucket}/{split_output_prefix}/test"
        ),
    ],
    code="code/data_split.py"
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


### 2.4. Model Training

In [23]:
import time

training_image = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.5-1"
)

framework_version="1.5-1"
xgb_estimator = sagemaker.estimator.Estimator(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region, framework_version),
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size=50,
    output_path=f"s3://{bucket}/final_project/model",
    sagemaker_session=sagemaker_session,
    hyperparameters={
        'num_round': 100,
        'objective': 'binary:logistic',
    }
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:botocore.credentials:Found credentials in environment variables.


In [24]:
train_input = sagemaker.inputs.TrainingInput(
    step_split_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
    content_type="text/csv"
)

validation_input = sagemaker.inputs.TrainingInput(
    step_split_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
    content_type="text/csv"
)

step_train_model = TrainingStep(
    name="ModelTraining",
    estimator=xgb_estimator,
    inputs={
        "train": train_input,
        "validation": validation_input
    }
)

### 2.5. Model Evaluation

## 3. Create Pipeline 

In [21]:
# Create the pipeline
pipeline = Pipeline(
    name="DDOS-Intrusion-Detection-Pipeline",
    parameters=[input_data_uri, feature_group_name],
    steps=[step_feature_engineering_process, step_feature_store_process, step_split_process, step_train_model],
    sagemaker_session=sagemaker_session
)

In [22]:
# Create or Update the Pipeline
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:249645693565:pipeline/DDOS-Intrusion-Detection-Pipeline',
 'ResponseMetadata': {'RequestId': 'e68aef45-8911-4c34-9394-842f92d2c74f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e68aef45-8911-4c34-9394-842f92d2c74f',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '101',
   'date': 'Fri, 13 Jun 2025 22:14:46 GMT'},
  'RetryAttempts': 0}}

## 4. Start Pipeline

In [23]:
# Start Pipeline Execution
execution = pipeline.start()
execution.wait()

In [24]:
print("Feature Engineered data location:", output_data_uri)

Feature Engineered data location: s3://sagemaker-us-east-1-249645693565/final_project/feature_engineer


In [25]:
sagemaker_client = boto3.client('sagemaker')
response = sagemaker_client.describe_pipeline_execution(
    PipelineExecutionArn=execution.arn
)

print("Pipeline failed reason:\n", response.get("FailureReason"))

Pipeline failed reason:
 None


In [26]:
execution.describe()

# See all steps and their status
for step in execution.list_steps():
    print(f"{step['StepName']}: {step['StepStatus']}")
    if step["StepStatus"] == "Failed":
        print("Failure reason:", step.get("FailureReason"))

FeatureIngest: Succeeded
FeatureEngineering: Succeeded


In [27]:
print("Pipeline execution completed. Execution ARN:", execution.arn)


Pipeline execution completed. Execution ARN: arn:aws:sagemaker:us-east-1:249645693565:pipeline/DDOS-Intrusion-Detection-Pipeline/execution/y4vjtle3mb92


## 5. Cleaup